 This notebook assumes you followed installation instructions in [Getting Started](getting-started.ipynb) or the [home page](../README.md). If you are running this in an online environment such as Colab, install the package first:

In [ ]:
!pip install aiondemand

# Working with ML Models on AI-on-Demand

 You can try this notebook interactively on [Google Colab](https://colab.research.google.com/github/aiondemand/aiondemand/blob/develop/docs/examples/models.ipynb) or [Binder](https://mybinder.org/v2/gh/aiondemand/aiondemand/develop?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fexamples%2Fmodels.ipynb).

This notebook shows you how to:
1. **Browse and search** ML models in the AIoD metadata catalogue
2. **Retrieve metadata** for a specific model
3. **Instantiate indexed models** (scikit-learn, XGBoost, etc.) directly from the SDK
4. **Register your own model** on AIoD so others can find it

For detailed written guides, see:
- [Using Models from AIoD](../guides/using-models.md)
- [Sharing Models to AIoD](../guides/sharing-models.md)

In [ ]:
import aiod
print(f"Using aiondemand v{aiod.__version__}")

## 1. Browsing ML Models

Let's start by seeing what models are available. The `get_list` function returns a pandas DataFrame by default.

In [ ]:
# Grab the first 10 models
models = aiod.ml_models.get_list(limit=10)
models[["name", "platform", "identifier"]].head()

You can also filter by platform. For example, to see only models from Hugging Face:

In [ ]:
hf_models = aiod.ml_models.get_list(platform="huggingface", limit=5)
hf_models[["name", "platform", "identifier"]].head()

### How Many Models Are There?

You can get a quick count without fetching all the data:

In [ ]:
total = aiod.ml_models.counts()
print(f"Total ML models in the catalogue: {total}")

# Break it down by platform
per_platform = aiod.ml_models.counts(per_platform=True)
per_platform

## 2. Searching for Models

If you know roughly what you're looking for, `search` is your best friend:

In [ ]:
results = aiod.ml_models.search("image classification")
results[["identifier", "name", "platform"]].head()

You can also search within a specific field, like the model name:

In [ ]:
results = aiod.ml_models.search("bert", search_field="name")
results[["identifier", "name", "platform"]].head()

## 3. Getting Model Details

Once you've found a model you're interested in, use its identifier to get the full metadata:

In [ ]:
# Let's pick the first model from our browse results
first_model = aiod.ml_models.get_list(limit=1, data_format="json")[0]
model_id = first_model["identifier"]
print(f"Looking up model: {model_id}")

# Get the full details
model = aiod.ml_models.get_asset(identifier=model_id, data_format="json")
print(f"Name:     {model['name']}")
print(f"Platform: {model['platform']}")
print(f"Keywords: {model.get('keyword', [])}")

## 4. Instantiating Indexed Models

AIoD indexes models from popular ML libraries like scikit-learn, XGBoost, and mlxtend. You can create instances of these models directly through the SDK — no need to figure out the right import path yourself.

Let's try it:

In [ ]:
# Get the RandomForestClassifier class from the AIoD registry
RandomForest = aiod.get("RandomForestClassifier")
print(f"Got: {RandomForest}")

# Create an instance with custom parameters
clf = RandomForest(n_estimators=50, random_state=42)
print(f"Created: {clf}")

Now let's use it on a real dataset — the classic Iris dataset:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train and evaluate
clf.fit(X_train, y_train)
accuracy = clf.score(X_test, y_test)
print(f"Test accuracy: {accuracy:.2%}")

### Building a Pipeline

You can even compose full pipelines using indexed components:

In [ ]:
# Get pipeline components from the registry
Scaler = aiod.get("StandardScaler")
PCA = aiod.get("PCA")
LogReg = aiod.get("LogisticRegression")
Pipe = aiod.get("Pipeline")

# Build the pipeline
pipeline = Pipe(steps=[
    ("scaler", Scaler()),
    ("pca", PCA(n_components=2)),
    ("classifier", LogReg(max_iter=200)),
])

# Train and evaluate
pipeline.fit(X_train, y_train)
print(f"Pipeline test accuracy: {pipeline.score(X_test, y_test):.2%}")

## 5. Sharing Your Own Model

To register a model on AIoD, you first need to authenticate. This requires an AIoD account — go to [aiod.eu](https://aiod.eu) and click **Login** to create one.

Then authenticate from Python:

In [ ]:
# Uncomment the lines below to authenticate
# aiod.create_token(write_to_file=True)
# aiod.get_current_user()

### Register a Model

Provide a dictionary of metadata. At minimum, include a `name`, but the more detail you add, the easier it will be for others to find and trust your model.

⚠️ **Important**: Metadata submitted to the catalogue is publicly visible. Please use real, meaningful data. Test entries will be removed, and repeated offenses may lead to a ban. Use a [local development server](https://github.com/aiondemand/aiod-rest-api) for testing.

In [ ]:
# Uncomment to register (requires authentication)

# identifier = aiod.ml_models.register(metadata={
#     "name": "My Iris Classifier",
#     "description": {
#         "plain": "A Random Forest classifier trained on the Iris dataset. "
#                  "Achieves ~96% test accuracy with 50 trees."
#     },
#     "keyword": ["iris", "classification", "random-forest", "scikit-learn"],
#     "license": "mit",
# })
# print(f"Registered! Identifier: {identifier}")

### Update and Delete

You can update a specific field without touching the rest:

In [ ]:
# Uncomment to update (requires authentication and a registered model)

# aiod.ml_models.update(
#     identifier=identifier,
#     metadata={"keyword": ["iris", "classification", "random-forest", "scikit-learn", "example"]},
# )
# print("Updated keywords!")

And if you need to remove a model you registered:

In [ ]:
# Uncomment to delete (requires authentication)
# ⚠️ This is permanent — there is no undo!

# aiod.ml_models.delete(identifier=identifier)
# print("Deleted.")

---

## What's Next?

- 📖 **Detailed guide: Using models** → [Using Models from AIoD](../guides/using-models.md)
- 📖 **Detailed guide: Sharing models** → [Sharing Models to AIoD](../guides/sharing-models.md)
- 📋 **API reference** → [ML Models API](../api/assets/ml_models.md)
- 🔐 **Authentication details** → [Authentication](../api/authentication.md)